In [29]:
from hyperflow.engine import Hyperflow
from hyperflow.ner import NOVEL_GLINER_LABELS

model_balanced = Hyperflow(
    save_dir="index_store/graphrag-bench-novel-balanced/Novel-44557",
    llm_model_name="gpt-4o-mini",
    use_reranker=False,
    gliner_labels=list(NOVEL_GLINER_LABELS),
    semantic_unit_percentile=60,  # Kamradt percentile 不变
    # min_words=20, max_words=200 是 create_semantic_units 的新默认值
)

INFO:hyperflow.engine:Initializing Hyperflow with config: HyperflowConfig(save_dir='index_store/graphrag-bench-novel-balanced/Novel-44557', embedding_model_name='BAAI/bge-large-en-v1.5', llm_model_name='gpt-4o-mini', query_instruction_prefix='Represent this sentence for searching relevant passages: ', chunk_token_size=1200, chunk_overlap_token_size=100, spacy_model='en_core_web_trf', batch_size=128, max_workers=16, retrieval_top_k=5, scoring_lambda=0.5, expansion_max_hops=3, expansion_top_k=15, hop_decay=0.5, conductance_floor=0.5, conductance_gamma=1.0, steering_alpha=0.7, steering_top_k=3, semantic_unit_percentile=60, use_reranker=False, reranker_model_name='Qwen/Qwen3-Reranker-4B', reranker_candidate_top_k=30, reranker_batch_size=8, reranker_max_length=4096, reranker_instruction='Given a multi-hop question, judge whether the document contains evidence that helps answer the question, either directly or as an intermediate bridge.', ner_backend='gliner', gliner_model='urchade/gliner_la

In [30]:
import pandas as pd
df = pd.read_parquet("index_store/graphrag-bench-novel/Novel-44557/passage_embedding.parquet")
passages = df['text'].tolist()
model_balanced.index(passages)

INFO:hyperflow.engine:NER cache status: cached_passages=0, uncached_passages=46
INFO:hyperflow.engine:Creating semantic units via Kamradt Percentile (p=60)...
Semantic Unit Chunking: 100%|██████████| 46/46 [00:16<00:00,  2.83it/s]
INFO:hyperflow.engine:Created 737 semantic units from 46 passages (avg 16.0 SU/passage)
Entity Extraction: 100%|██████████| 46/46 [00:54<00:00,  1.19s/it]
INFO:hyperflow.entity_normalization:Entity merge: 20 entities merged into 20 groups (threshold=0.93)


In [31]:
import numpy as np

model_balanced._prepare_retrieval_cache()

query = "What is the connection between Kynance and Cornwall as established by the narrative chains?"
query_embedding = model_balanced.embedding_model.encode(
    query, normalize_embeddings=True, prompt=model_balanced.config.query_instruction_prefix
)

# Seed entities
query_entities, seed_indices, seed_texts, seed_hash_ids, seed_scores = \
    model_balanced.extract_seed_entities(query)
print("Query NER:", query_entities)
print("Seed entities:", list(zip(seed_texts, seed_scores)))

# Frontier expansion
from hyperflow.frontier import frontier_expansion
activated_entities = frontier_expansion(
    config=model_balanced.config,
    hypergraph=model_balanced.knowledge_graph._hypergraph,
    entity_hash_ids=model_balanced._entity_hash_ids,
    su_embeddings=model_balanced._su_embeddings,
    question_embedding=query_embedding,
    seed_entity_indices=seed_indices,
    seed_entity_hash_ids=seed_hash_ids,
    seed_entity_scores=seed_scores,
    entity_embeddings=model_balanced._entity_embeddings,
)

print("\nActivated entities:")
for h_id, (idx, score) in sorted(activated_entities.items(), key=lambda x: -x[1][1]):
    text = model_balanced.entity_embedding_store.hash_id_to_text[h_id]
    print(f"  {score:.4f}  {text}")



INFO:hyperflow.engine:Building hypergraph...
INFO:hyperflow.knowledge_graph:Hypergraph built: 1461 vertices, 671 hyperedges, nnz=2665, sparsity=99.73%, D_v range [1, 53], D_e range [1, 20]
Batches: 100%|██████████| 1/1 [00:00<00:00, 279.62it/s]
INFO:hyperflow.frontier:Conductance (floor=0.50, gamma=1.00): 51/671 active SUs, 620 suppressed
INFO:hyperflow.frontier:Progressive steering enabled: alpha=0.70, top_k=3
DEBUG:hyperflow.frontier:Hop 1: 15 new entities (top score=0.1771, decay=0.500), 18 total activated
DEBUG:hyperflow.frontier:Steering after hop 1: 189 active SUs (was 51)
DEBUG:hyperflow.frontier:Hop 2: 15 new entities (top score=0.0257, decay=0.250), 33 total activated
DEBUG:hyperflow.frontier:Steering after hop 2: 189 active SUs (was 51)
DEBUG:hyperflow.frontier:Hop 3: 15 new entities (top score=0.0021, decay=0.125), 48 total activated
DEBUG:hyperflow.frontier:Steering after hop 3: 189 active SUs (was 51)
INFO:hyperflow.frontier:Frontier expansion finished: 3 hops, 48 activate

Query NER: ['kynance', 'narrative chain', 'cornwall']
Seed entities: [('kynance', 0.90717393), ('this tale', 0.5900511), ('cornwall', 0.8084472)]

Activated entities:
  0.9072  kynance
  0.8084  cornwall
  0.5901  this tale
  0.0885  sennen cove
  0.0885  lion rock
  0.0885  trevena
  0.0885  boscastle
  0.0885  enys dodnan and pardenick point
  0.0885  st. ive
  0.0885  kynance cove
  0.0885  old post office
  0.0885  steeple rock
  0.0885  cornish fisherman
  0.0885  seine boat
  0.0885  john curgenven
  0.0885  tintagel
  0.0650  bathing gown
  0.0650  rock parlour
  0.0064  gorlois
  0.0064  tintagel castle
  0.0064  king arthur
  0.0064  elizabeth
  0.0064  ygrayne
  0.0064  castle terrabil
  0.0060  ristormel
  0.0060  bodmin road
  0.0060  king mark
  0.0060  marazion
  0.0060  st. ives bay
  0.0060  redruth
  0.0060  st. austell
  0.0060  grampound
  0.0060  king arthurs
  0.0003  giant cormoran
  0.0003  ph[oe]nicians
  0.0003  goonhilly down
  0.0003  trelowarren
  0.0003  ja

In [32]:
sorted_hash_ids, sorted_scores = model_balanced._diffuse_from_seeds(
    query, query_embedding, seed_indices, seed_hash_ids, seed_scores
)

print("=== Top 5 Passages ===")
for i in range(5):
    h_id = sorted_hash_ids[i]
    text = model_balanced.passage_embedding_store.hash_id_to_text[h_id]
    print(f"\n--- Passage {i+1} (score={sorted_scores[i]:.4f}) ---")
    print(text[:200] + "...")

# 检查 Devil's Throat passages 排名
print("\n=== Devil's Throat passages ===")
for h_id in sorted_hash_ids:
    text = model_balanced.passage_embedding_store.hash_id_to_text[h_id]
    if "Devil" in text and "Throat" in text:
        rank = sorted_hash_ids.index(h_id) + 1
        print(f"  rank={rank}  score={sorted_scores[rank-1]:.4f}")
        print(f"  {text[:150]}...")


INFO:hyperflow.frontier:Conductance (floor=0.50, gamma=1.00): 51/671 active SUs, 620 suppressed
INFO:hyperflow.frontier:Progressive steering enabled: alpha=0.70, top_k=3
DEBUG:hyperflow.frontier:Hop 1: 15 new entities (top score=0.1771, decay=0.500), 18 total activated
DEBUG:hyperflow.frontier:Steering after hop 1: 189 active SUs (was 51)
DEBUG:hyperflow.frontier:Hop 2: 15 new entities (top score=0.0257, decay=0.250), 33 total activated
DEBUG:hyperflow.frontier:Steering after hop 2: 189 active SUs (was 51)
DEBUG:hyperflow.frontier:Hop 3: 15 new entities (top score=0.0021, decay=0.125), 48 total activated
DEBUG:hyperflow.frontier:Steering after hop 3: 189 active SUs (was 51)
INFO:hyperflow.frontier:Frontier expansion finished: 3 hops, 48 activated entities


=== Top 5 Passages ===

--- Passage 1 (score=0.9983) ---
Produced by Chris Curnow and the Online Distributed Proofreading Team at http://www.pgdp.net (This file was produced from images generously made available by The Internet Archive) AN UNSENTIMENTAL JOU...

--- Passage 2 (score=0.8548) ---
Not however without taking a long look out of the window upon the bay, which now, at high tide, was one sheet of rippling moon-lit water, with the grim old Mount, full of glimmering lights like eyes, ...

--- Passage 3 (score=0.7254) ---
The girls had found a way, chiefly on the tops of "hedges," to the grand rock called Lizard Point. Thither we went, and watched the sunset--a very fine one; then came back through the village, and mad...

--- Passage 4 (score=0.7249) ---
[Illustration: KYNANCE COVE, CORNWALL.] "Ower the muir amang the heather" have I tramped many a mile in bonnie Scotland, but this Cornish moor and Cornish heather were quite different. As different as...

--- Passage 5 (score=0.7

In [33]:
top_passages = [model_balanced.passage_embedding_store.hash_id_to_text[h] for h in sorted_hash_ids[:5]]

system_prompt = (
    "As an advanced reading comprehension assistant, your task is to analyze text passages "
    "and corresponding questions meticulously. Your response start after \"Thought: \", "
    "where you will methodically break down the reasoning process, illustrating how you "
    "arrive at conclusions. Conclude with \"Answer: \" to present a concise, definitive "
    "response, devoid of additional elaborations."
)

prompt_user = ""
for p in top_passages:
    prompt_user += f"{p}\n"
prompt_user += f"Question: {query}\n Thought: "

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt_user}
]

response = model_balanced.llm_model.infer(messages)
print(response)

DEBUG:httpx:load_ssl_context verify=True cert=None trust_env=False http2=False
DEBUG:httpx:load_verify_locations cafile='/home/coder/Hyperflow/venv/lib/python3.10/site-packages/certifi/cacert.pem'
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'system', 'content': 'As an advanced reading comprehension assistant, your task is to analyze text passages and corresponding questions meticulously. Your response start after "Thought: ", where you will methodically break down the reasoning process, illustrating how you arrive at conclusions. Conclude with "Answer: " to present a concise, definitive response, devoid of additional elaborations.'}, {'role': 'user', 'content': 'Produced by Chris Curnow and the Online Distributed Proofreading Team at http://www.pgdp.net (This file was produced from images generously made available by The Internet Archive) AN UNSENTIMENTAL JOURNEY THROUGH CORNWALL [Illustrat

Thought: The narrative establishes a strong connection between Kynance and Cornwall through the author's detailed descriptions of the landscape, local customs, and the experiences of the travelers. Kynance is depicted as a beautiful and unique location within Cornwall, characterized by its stunning natural features such as the cliffs, coves, and the sea. The author emphasizes the local culture, including the fishing practices and the community's way of life, which are integral to the identity of Cornwall. Additionally, the interactions with local characters, like John Curgenven, further anchor Kynance within the broader context of Cornish heritage and tradition. The narrative also highlights the emotional attachment the travelers develop towards Kynance, reflecting the allure and charm of Cornwall as a whole. Thus, Kynance serves as a microcosm of the larger Cornish experience, illustrating the region's natural beauty, cultural richness, and the warmth of its people.

Answer: Kynance i